# IMQCAM API Access for Modelers

This notebook demonstrates how to use the IMQCAM API to find and access files programmatically.

The main workflow is:

1. Connect to IMQCAM.
2. Choose an IGSN and data type.
3. Query the `imqcam/datafiles` endpoint.
4. Retrieve metadata such as composition.
5. Review the matching files.
6. Download files when needed.

## Key concepts

**API**  
An API allows Python code to communicate directly with the IMQCAM data portal without manually browsing through folders in the web interface.

**Endpoint**  
An endpoint is a specific API location used for a particular type of request. In this notebook, we use:

`imqcam/datafiles`

This endpoint is used to retrieve data-file records from IMQCAM.

**GET request**  
A GET request asks the API to return information. It does not modify or delete data.

**Parameters**  
Parameters tell the API what data we want. Examples include the data type, IGSN, pagination limit, and metadata fields.

**Metadata**  
Metadata describes the scientific data. Examples include IGSN, composition, file name, data type, and other descriptive information.


## Before you start

Keep this notebook in the IMQCAM project/repository where `imqcam.py` is available.

The helper file `imqcam.py` handles the connection to the IMQCAM API. If the imports below fail, confirm that `imqcam.py` is in this folder or one of its parent folders.

If needed, install the supporting packages in the same Python/Jupyter environment:

```python
%pip install pandas python-dotenv girder-client

## 1. Connect to IMQCAM

This cell creates an authenticated IMQCAM client.

The client knows the IMQCAM API base URL and is used for all later API requests.


In [2]:
import json
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client, fetch_entries, entries_to_frame, extract_igsn

client = get_client()
print(f"Connected to {client.urlBase}")

Connected to https://data.imqcam.org/api/v1/


## 2. Choose the specimen and data type

The two most important inputs are:

- **IGSN** — identifies the specimen or sample.
- **DATA_TYPE** — identifies the type of scientific data to retrieve, such as EBSD raw data.

Modelers normally only need to change these values for a different specimen or dataset.


In [3]:
IGSN = "CMXMAG00001-039"
DATA_TYPE = "ebsd_raw"

# Optional local filter after the API results are returned.
# Examples: "IN718", "Ti64"
# Use None if you do not want to filter by composition.
COMPOSITION_FILTER = None

print("IGSN:", IGSN)
print("Data type:", DATA_TYPE)
print("Composition filter:", COMPOSITION_FILTER or "None")

IGSN: CMXMAG00001-039
Data type: ebsd_raw
Composition filter: None


## 3. Query the data-files endpoint

This cell sends a GET request to:

`imqcam/datafiles`

The request asks IMQCAM for records that match the selected specimen and data type.

### What the parameters mean

- `dataType` — the type of data requested.
- `meta.igsn` — filters the results to the selected specimen.
- `extraFields` — asks the API to return additional metadata, such as composition.
- `limit` — controls how many records are returned in one request.
- `offset` — is used for pagination when there are more records than one request can return.

In plain language, the request means:

> Give me the matching IMQCAM data files for this specimen and data type, and include composition metadata.


In [4]:
def get_all_files(client, igsn, data_type, page_size=100):
    all_items = []
    offset = 0

    while True:
        batch = client.get(
            "imqcam/datafiles",
            parameters={
                "dataType": data_type,
                "filters": json.dumps({
                    "meta.igsn": igsn
                }),
                "extraFields": json.dumps([
                    "meta.composition"
                ]),
                "limit": page_size,
                "offset": offset,
            },
        )

        all_items.extend(batch)
        print(
            f"Offset {offset:,}: received {len(batch):,} | "
            f"total collected {len(all_items):,}"
        )

        # A short page means we reached the final page.
        if len(batch) < page_size:
            break

        offset += page_size

    return all_items


## 2. Choose the specimen and data type

The two most important inputs are:

- **IGSN** — identifies the specimen or sample.
- **DATA_TYPE** — identifies the type of scientific data to retrieve, such as EBSD raw data.

Modelers normally only need to change these values for a different specimen or dataset.


In [5]:
items = get_all_files(
    client=client,
    igsn=IGSN,
    data_type=DATA_TYPE,
)

print(f"\nTotal matching API records: {len(items):,}")

Offset 0: received 100 | total collected 100
Offset 100: received 100 | total collected 200
Offset 200: received 100 | total collected 300
Offset 300: received 100 | total collected 400
Offset 400: received 100 | total collected 500
Offset 500: received 100 | total collected 600
Offset 600: received 100 | total collected 700
Offset 700: received 100 | total collected 800
Offset 800: received 100 | total collected 900
Offset 900: received 100 | total collected 1,000
Offset 1,000: received 100 | total collected 1,100
Offset 1,100: received 100 | total collected 1,200
Offset 1,200: received 100 | total collected 1,300
Offset 1,300: received 100 | total collected 1,400
Offset 1,400: received 100 | total collected 1,500
Offset 1,500: received 100 | total collected 1,600
Offset 1,600: received 100 | total collected 1,700
Offset 1,700: received 100 | total collected 1,800
Offset 1,800: received 100 | total collected 1,900
Offset 1,900: received 100 | total collected 2,000
Offset 2,000: receiv

## 5. Review metadata and composition

The actual file contains the scientific measurement data.

Metadata describes that file or specimen. In this notebook, one important metadata field is:

`meta.composition`

For example, composition metadata may identify the material associated with a specimen.

This is useful for modelers because they can confirm that the retrieved files belong to the expected material before downloading or processing them.


In [6]:
preview_rows = []

for item in items:
    meta = item.get("meta", {}) or {}

    preview_rows.append({
        "file_name": item.get("name"),
        "igsn": meta.get("igsn"),
        "composition": meta.get("composition"),
        "item_id": item.get("_id"),
        "size": item.get("size"),
    })

files_df = pd.DataFrame(preview_rows)

print(f"Records in table: {len(files_df):,}")
display(files_df.head(20))

Records in table: 6,247


,file_name,igsn,composition,item_id,size
0,00116eef-595d-44fb-b9a1-0fa7506dd3fb.dat,CMXMAG00001-039,IN718,6aa324334361c9b31422aec7,1966080
1,001b18d7-2a05-40ea-b115-68d85a7bd616.dat,CMXMAG00001-039,IN718,6aa3237f58a0ff9bfab5218e,5701650
2,0035fdd6-5afa-435a-b073-8c7db66c56a6.dat,CMXMAG00001-039,IN718,6aa32429018f487e6ce58cfc,1966080
3,0036210e-eb2c-4909-b839-9abf2497b7ce.dat,CMXMAG00001-039,IN718,6aa324524361c9b31422aecb,1966080
4,00474464-dd88-4291-a6e7-b058bf5773d4.dat,CMXMAG00001-039,IN718,6aa3246158a0ff9bfab5219e,1966080
5,005cd894-b18b-4400-bbbe-2751f186412c.dat,CMXMAG00001-039,IN718,6aa323ba018f487e6ce58ce3,196608
6,00864ff4-bac0-4984-bb3f-a2307cd4800f.dat,CMXMAG00001-039,IN718,6aa3243d018f487e6ce58d01,786432
7,009283dd-44eb-41be-a697-02d47a55605a.dat,CMXMAG00001-039,IN718,6aa3244d018f487e6ce58d07,1966080
8,00aeb2f6-5826-4fb6-86ba-b34a55bb52db.dat,CMXMAG00001-039,IN718,6aa32317018f487e6ce58cc8,786432
9,00b82d9e-16be-4bf4-bc3b-3f861ba80727.dat,CMXMAG00001-039,IN718,6aa32320018f487e6ce58ccb,1966080


## 2. Choose the specimen and data type

The two most important inputs are:

- **IGSN** — identifies the specimen or sample.
- **DATA_TYPE** — identifies the type of scientific data to retrieve, such as EBSD raw data.

Modelers normally only need to change these values for a different specimen or dataset.


In [ ]:
DOWNLOAD_FILES = False
DOWNLOAD_LIMIT = 5

if DOWNLOAD_FILES:
    selected_ids = set(filtered_df["item_id"].dropna())

    items_to_download = [
        item for item in items
        if item.get("_id") in selected_ids
    ]

    if DOWNLOAD_LIMIT is not None:
        items_to_download = items_to_download[:DOWNLOAD_LIMIT]

    download_folder = REPO_ROOT / "downloads" / DATA_TYPE / IGSN
    download_folder.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {len(items_to_download):,} item(s)...")
    print(f"Destination: {download_folder}")

    for number, item in enumerate(items_to_download, start=1):
        client.downloadItem(
            item["_id"],
            dest=str(download_folder)
        )

        print(
            f"[{number}/{len(items_to_download)}] "
            f"{item.get('name', item['_id'])}"
        )

    print("Download complete.")

else:
    print(
        "Downloads are OFF. Review the results first, then set "
        "DOWNLOAD_FILES = True when ready."
    )


Destination: /Users/akhilaponugoti/Documents/imqcam_dms_api/downloads/ebsd_raw/CMXMAG00001-039
[1/5] 00116eef-595d-44fb-b9a1-0fa7506dd3fb.dat
[2/5] 001b18d7-2a05-40ea-b115-68d85a7bd616.dat
[3/5] 0035fdd6-5afa-435a-b073-8c7db66c56a6.dat
[4/5] 0036210e-eb2c-4909-b839-9abf2497b7ce.dat
[5/5] 00474464-dd88-4291-a6e7-b058bf5773d4.dat
Download complete.


## Summary for modelers

The API workflow can be remembered as:

**IGSN → Data Type → API Endpoint → Metadata → Matching Files → Download**

### Important terms

- **API** — allows Python to communicate with IMQCAM.
- **Endpoint** — the specific API resource being requested.
- **GET** — retrieves information.
- **IGSN** — identifies the specimen.
- **Data Type** — identifies the category of scientific data.
- **Metadata** — describes the data, such as composition.
- **Pagination** — retrieves large result sets in multiple batches.

You do not need to become an API developer to use this workflow. In most cases, you only need to provide the specimen IGSN and the data type, then run the notebook cells in order.
